# C2 · Matched pilot

Ba arms dùng chung warmup checkpoint và $\tau_{\mathrm{tr}}$ cho mỗi seed. Giữ cùng config, loader, augmentation và optimizer budget. `profile.json` tổng hợp tối đa 500 updates đầu, gồm throughput, VRAM, epoch timing và diagnostics của oracle.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive")
REPO = DRIVE / "C2_Base"
WORK = DRIVE / "C2_work"
DATA_ROOT = Path("/content/digiface72")  # Extracted, authorized 72-capture partition
MANIFEST = WORK / "manifests/digiface72.csv"

assert (REPO / "configs/pilot.yaml").is_file(), f"Set REPO to the copied C2_Base source folder: {REPO}"
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
os.environ.setdefault("OMP_NUM_THREADS", "2")

def run(module, *args):
    subprocess.run([sys.executable, "-m", module, *map(str, args)], cwd=REPO, check=True)

In [ ]:
import torch

DEVICE = "cuda:0"
assert torch.cuda.is_available(), "Select a Colab GPU runtime before training."
print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory / 2**30, "GiB")

In [ ]:
CONFIG = REPO / "configs/pilot.yaml"
SCOPE = "pilot"
SEEDS = [17]  # Multi-seed screen: [17, 37, 61]
AMP_MODE = "off"
assert MANIFEST.is_file() and DATA_ROOT.is_dir()

if AMP_MODE == "bf16":
    assert torch.cuda.is_bf16_supported(), "This GPU does not support bfloat16."
if AMP_MODE != "off":
    run("pytest", "-q", "tests/test_models.py", "-m", "cuda", "-k", AMP_MODE)

In [ ]:
for seed in SEEDS:
    run_root = WORK / "runs" / SCOPE / f"seed-{seed}"
    shared = run_root / "warmup"
    assert (shared / "tau_train.json").is_file(), f"Finish common warmup for seed {seed}."
    for arm in ("nominal", "row", "c2"):
        out = run_root / arm
        resume = ["--resume"] if (out / "last.pt").exists() else []
        run("scripts.train", arm, "--config", CONFIG, "--warmup-dir", shared,
            "--manifest", MANIFEST, "--data-root", DATA_ROOT, "--out", out,
            "--scope", SCOPE, "--seed", seed, "--amp", AMP_MODE, "--device", DEVICE, *resume)
    run("scripts.report_runs", "profile", "--run-root", run_root, "--limit", 500,
        "--output", run_root / "profile.json")

In [ ]:
for seed in SEEDS:
    profile_path = WORK / "runs" / SCOPE / f"seed-{seed}" / "profile.json"
    profiling = json.loads(profile_path.read_text(encoding="utf-8"))
    for arm, details in profiling.items():
        print(seed, arm, {key: value for key, value in details.items() if key != "epochs"})